# ANALISIS PENJUALAN BULANAN PT MENSANA ANEKA SATWA
## Q1 2026 - Januari s/d Maret 2026

---

## SOAL

PT. Mensana memiliki 10 cabang distributor yang tersebar luas di seluruh Indonesia. 
Terdapat 15 produk unggulan yang dijual oleh masing-masing cabang distributor. Namun, 
data nama produk dan kode produk tiap cabang tidaklah sama (berbeda seluruhnya).  

### Yang harus dikerjakan:
1. Buatlah data mentah 3 bulan penjualan cabang
2. Buatlah laporan siap presentasi (Input, Proses, Output)
3. Total omset, progress tiap cabang, produk terlaris, rekomendasi
4. Visualisasi data semaksimal mungkin
5. Waktu pengerjaan 24 jam

In [ ]:
# ============================================================
# INPUT: Load Data & Agregasi Bulanan
# ============================================================
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings, os
warnings.filterwarnings('ignore')

os.makedirs('charts', exist_ok=True)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Load data harian
df_harian = pd.read_excel('Data_Mentah_Harian.xlsx', sheet_name='Data_Penjualan_Harian')
df_mapping = pd.read_excel('Data_Mentah_Harian.xlsx', sheet_name='Mapping_Produk')
df_harian['Tanggal'] = pd.to_datetime(df_harian['Tanggal'])

month_order = ['Januari', 'Februari', 'Maret']
day_order = ['Senin', 'Selasa', 'Rabu', 'Kamis', 'Jumat', 'Sabtu', 'Minggu']

# ============ AGREGASI BULANAN ============
df_bulanan = df_harian.groupby(['Bulan', 'ID_Cabang', 'Nama_Cabang', 'Provinsi']).agg(
    Total_Kuantiti=('Kuantiti', 'sum'),
    Total_Omset=('Omset', 'sum'),
    Hari_Aktif=('Tanggal', 'nunique')
).reset_index()
df_bulanan['Bulan'] = pd.Categorical(df_bulanan['Bulan'], categories=month_order, ordered=True)
df_bulanan = df_bulanan.sort_values(['Bulan', 'Total_Omset'], ascending=[True, False])

# Agregasi bulanan per produk
df_produk_bulanan = df_harian.groupby(['Bulan', 'Nama_Standar', 'Kategori']).agg(
    Total_Kuantiti=('Kuantiti', 'sum'),
    Total_Omset=('Omset', 'sum')
).reset_index()
df_produk_bulanan['Bulan'] = pd.Categorical(df_produk_bulanan['Bulan'], categories=month_order, ordered=True)

# Agregasi bulanan per kategori
df_kategori_bulanan = df_harian.groupby(['Bulan', 'Kategori']).agg(
    Total_Kuantiti=('Kuantiti', 'sum'),
    Total_Omset=('Omset', 'sum')
).reset_index()
df_kategori_bulanan['Bulan'] = pd.Categorical(df_kategori_bulanan['Bulan'], categories=month_order, ordered=True)

# Total omset
total_omset = df_harian['Omset'].sum()
total_qty = df_harian['Kuantiti'].sum()

print('='*70)
print('DATA BULANAN PT MENSANA ANEKA SATWA - Q1 2026')
print('='*70)
print(f'Total Omset 3 Bulan : Rp {total_omset:,.0f}')
print(f'Total Kuantiti      : {total_qty:,} unit')
print(f'Rata-rata/Bulan     : Rp {total_omset/3:,.0f}')
print(f'Jumlah Cabang       : {df_harian["ID_Cabang"].nunique()}')
print(f'Jumlah Produk       : {df_harian["Nama_Standar"].nunique()}')

---
## 1. INPUT - Data Mentah

Data penjualan 10 cabang distributor selama 3 bulan, diagregasi menjadi data bulanan.

**Masalah:** Setiap cabang menggunakan kode dan nama produk yang berbeda-beda.

**Solusi:** Tabel mapping menghubungkan kode per cabang ke kode standar perusahaan.

In [ ]:
# Contoh Mapping Produk
print('Contoh Mapping Produk (Supralit di 3 cabang):\n')
for _, row in df_mapping.head(1).iterrows():
    print(f'Kode Standar : {row["Kode_Standar"]} - {row["Nama_Standar"]}')
    print(f'Kategori     : {row["Kategori"]}')
    print(f'Harga Standar: Rp {row["Harga_Standar"]:,.0f}')
    print('\nKode di masing-masing cabang:')
    for cab in ['CAB-01', 'CAB-02', 'CAB-03', 'CAB-04', 'CAB-05']:
        print(f'  {cab}: {row[f"{cab}_Kode"]:15s} = {row[f"{cab}_Nama"]}')
    print('-'*50)

---
## 2. OUTPUT - Total Omset 3 Bulan

In [ ]:
# Ringkasan Total Omset
print('='*70)
print('TOTAL OMSET SELURUH CABANG - Q1 2026')
print('='*70)
print(f'\nTotal Omset 3 Bulan    : Rp {total_omset:>20,.0f}')
print(f'Total Kuantiti         : {total_qty:>20,} unit')
print(f'Rata-rata Omset/Bulan  : Rp {total_omset/3:>20,.0f}')
print(f'Rata-rata Omset/Cabang : Rp {total_omset/10/3:>20,.0f}')

# Ringkasan per bulan
print('\n' + '-'*70)
print('RINGKASAN PER BULAN:')
print('-'*70)
monthly = df_harian.groupby('Bulan').agg(
    Total_Omset=('Omset', 'sum'),
    Total_Kuantiti=('Kuantiti', 'sum'),
    Hari=('Tanggal', 'nunique')
).reset_index()
monthly['Bulan'] = pd.Categorical(monthly['Bulan'], categories=month_order, ordered=True)
monthly = monthly.sort_values('Bulan')
monthly['Rata_rata_Harian'] = (monthly['Total_Omset'] / monthly['Hari']).round(0)

for _, row in monthly.iterrows():
    print(f"{row['Bulan']:10s}: Rp {row['Total_Omset']:>15,.0f} | {row['Total_Kuantiti']:>6,} unit | Rata2/Hari: Rp {row['Rata_rata_Harian']:>12,.0f}")

---
## 3. ANALISIS PER BULAN

In [ ]:
# Growth Rate Bulanan
print('='*70)
print('PERTUMBUHAN BULANAN (GROWTH RATE)')
print('='*70)

monthly_omset = monthly.set_index('Bulan')['Total_Omset']
growth_jan_feb = ((monthly_omset['Februari'] - monthly_omset['Januari']) / monthly_omset['Januari'] * 100)
growth_feb_mar = ((monthly_omset['Maret'] - monthly_omset['Februari']) / monthly_omset['Februari'] * 100)

print(f'\nJanuari  -> Februari : {growth_jan_feb:+.1f}%')
print(f'Februari -> Maret    : {growth_feb_mar:+.1f}%')
print(f'\nTotal Q1             : Rp {total_omset:,.0f}')

# Visualisasi pertumbuhan bulanan
fig_growth = px.bar(
    x=['Januari', 'Februari', 'Maret'],
    y=[monthly_omset['Januari'], monthly_omset['Februari'], monthly_omset['Maret']],
    title='Total Omset per Bulan',
    labels={'x': 'Bulan', 'y': 'Total Omset (Rp)'},
    text_auto='.2s'
)
fig_growth.update_layout(height=400, showlegend=False)
fig_growth.show()

---
## 4. PROGRESS OMSET TIAP CABANG

In [ ]:
# Progress per cabang per bulan
print('='*70)
print('PROGRESS OMSET TIAP CABANG')
print('='*70)

progress = df_harian.groupby(['Nama_Cabang', 'Bulan']).agg(
    Omset=('Omset', 'sum')
).reset_index()
progress_pivot = progress.pivot(index='Nama_Cabang', columns='Bulan', values='Omset')
progress_pivot = progress_pivot[['Januari', 'Februari', 'Maret']]
progress_pivot['Total_Q1'] = progress_pivot.sum(axis=1)
progress_pivot['Growth_Jan_Feb'] = ((progress_pivot['Februari'] - progress_pivot['Januari']) / progress_pivot['Januari'] * 100).round(1)
progress_pivot['Growth_Feb_Mar'] = ((progress_pivot['Maret'] - progress_pivot['Februari']) / progress_pivot['Februari'] * 100).round(1)
progress_pivot = progress_pivot.sort_values('Total_Q1', ascending=False)

print('\nPROGRESS OMSET TIAP CABANG (Rp):\n')
progress_pivot.style.format('{:,.0f}', subset=['Januari', 'Februari', 'Maret', 'Total_Q1']).format('{:.1f}%', subset=['Growth_Jan_Feb', 'Growth_Feb_Mar'])

In [ ]:
# Visualisasi progress
progress_reset = progress_pivot.reset_index()
progress_melted = progress_reset.melt(id_vars='Nama_Cabang', value_vars=['Januari', 'Februari', 'Maret'], 
                                      var_name='Bulan', value_name='Omset')
progress_melted['Bulan'] = pd.Categorical(progress_melted['Bulan'], categories=month_order, ordered=True)

fig_progress = px.bar(
    progress_melted,
    x='Nama_Cabang',
    y='Omset',
    color='Bulan',
    barmode='group',
    title='Progress Omset per Cabang per Bulan',
    labels={'Omset': 'Omset (Rp)', 'Nama_Cabang': 'Cabang'}
)
fig_progress.update_layout(height=500)
fig_progress.show()

---
## 5. RANKING CABANG

In [ ]:
# Ranking cabang
branch_ranking = df_harian.groupby(['ID_Cabang', 'Nama_Cabang', 'Provinsi']).agg(
    Total_Omset=('Omset', 'sum'),
    Total_Kuantiti=('Kuantiti', 'sum'),
    Rata_rata_Harian=('Omset', 'mean')
).reset_index().sort_values('Total_Omset', ascending=False)

branch_ranking['Persentase'] = (branch_ranking['Total_Omset'] / total_omset * 100).round(2)
branch_ranking['Ranking'] = range(1, len(branch_ranking) + 1)
branch_ranking['Rata_rata_Bulanan'] = branch_ranking['Total_Omset'] / 3

# Segmentasi
def segmentasi(rank):
    if rank <= 2:
        return 'Top Performer'
    elif rank <= 6:
        return 'Good'
    else:
        return 'Needs Improvement'

branch_ranking['Segmen'] = branch_ranking['Ranking'].apply(segmentasi)

print('='*70)
print('RANKING CABANG Q1 2026')
print('='*70)
print('\n' + '-'*100)
for _, row in branch_ranking.iterrows():
    print(f"#{int(row['Ranking']):2d} | {row['Nama_Cabang']:15s} | Rp {row['Total_Omset']:>15,.0f} | {row['Persentase']:5.1f}% | Rata2/Bulan: Rp {row['Rata_rata_Bulanan']:>12,.0f} | {row['Segmen']}")

print('\n' + '='*70)
print('SEGMENTASI CABANG:')
print('='*70)
for segmen in ['Top Performer', 'Good', 'Needs Improvement']:
    subset = branch_ranking[branch_ranking['Segmen'] == segmen]
    print(f'\n{segmen}:')
    for _, row in subset.iterrows():
        print(f'  - {row["Nama_Cabang"]}: Rp {row["Total_Omset"]:,.0f} ({row["Persentase"]:.1f}%)')

---
## 6. PRODUK TERLARIS

In [ ]:
# Ranking produk
print('='*70)
print('RANKING 15 PRODUK TERLARIS Q1 2026')
print('='*70)

product_ranking = df_harian.groupby(['Nama_Standar', 'Kategori']).agg(
    Total_Omset=('Omset', 'sum'),
    Total_Kuantiti=('Kuantiti', 'sum')
).reset_index().sort_values('Total_Omset', ascending=False)
product_ranking['Persentase'] = (product_ranking['Total_Omset'] / total_omset * 100).round(2)
product_ranking['Ranking'] = range(1, len(product_ranking) + 1)
product_ranking['Rata_rata_Bulanan'] = product_ranking['Total_Omset'] / 3

print('\n' + '-'*100)
for _, row in product_ranking.iterrows():
    print(f"#{int(row['Ranking']):2d} | {row['Nama_Standar']:20s} | {row['Kategori']:15s} | Rp {row['Total_Omset']:>15,.0f} | {row['Persentase']:5.1f}%")

# Top 5 vs Bottom 5
top5_total = product_ranking.head(5)['Total_Omset'].sum()
bottom5_total = product_ranking.tail(5)['Total_Omset'].sum()
print(f'\nTop 5 Produk  : Rp {top5_total:>15,.0f} ({top5_total/total_omset*100:.1f}%)')
print(f'Bottom 5 Produk: Rp {bottom5_total:>15,.0f} ({bottom5_total/total_omset*100:.1f}%)')

---
## 7. ANALISIS KATEGORI

In [ ]:
# Distribusi kategori
print('='*70)
print('DISTRIBUSI OMSET PER KATEGORI Q1 2026')
print('='*70)

category_summary = df_harian.groupby('Kategori').agg(
    Total_Omset=('Omset', 'sum'),
    Total_Kuantiti=('Kuantiti', 'sum'),
    Jumlah_Produk=('Nama_Standar', 'nunique')
).reset_index().sort_values('Total_Omset', ascending=False)
category_summary['Persentase'] = (category_summary['Total_Omset'] / total_omset * 100).round(2)
category_summary['Rata_rata_Bulanan'] = category_summary['Total_Omset'] / 3
category_summary['Rata_rata_per_Produk'] = category_summary['Total_Omset'] / category_summary['Jumlah_Produk']

print('\n' + '-'*100)
for _, row in category_summary.iterrows():
    print(f"{row['Kategori']:20s} | Rp {row['Total_Omset']:>15,.0f} | {row['Persentase']:5.1f}% | {row['Jumlah_Produk']:2d} produk | Rata2/Produk: Rp {row['Rata_rata_per_Produk']:>12,.0f}")

# Tren kategori per bulan
print('\n' + '='*70)
print('TREN KATEGORI PER BULAN:')
print('='*70)

cat_monthly = df_kategori_bulanan.pivot(index='Kategori', columns='Bulan', values='Total_Omset')
cat_monthly = cat_monthly[['Januari', 'Februari', 'Maret']]
cat_monthly['Growth_Feb_Mar'] = ((cat_monthly['Maret'] - cat_monthly['Februari']) / cat_monthly['Februari'] * 100).round(1)

print('\n' + cat_monthly.to_string())

---
## 8. POLA HARI KERJA vs AKHIR PEKAN

In [ ]:
# Hari kerja vs weekend
print('='*70)
print('POLA HARI KERJA vs AKHIR PEKAN')
print('='*70)

df_harian['Tipe_Hari'] = df_harian['Hari'].apply(
    lambda x: 'Akhir Pekan' if x in ['Sabtu', 'Minggu'] else 'Hari Kerja'
)

dow_comparison = df_harian.groupby('Tipe_Hari').agg(
    Total_Omset=('Omset', 'sum'),
    Rata_rata=('Omset', 'mean'),
    Jumlah=('Omset', 'count')
).reset_index()

print('\nPerbandingan:')
for _, row in dow_comparison.iterrows():
    print(f"{row['Tipe_Hari']:15s}: Rata-rata Rp {row['Rata_rata']:>12,.0f}/hari | Total: Rp {row['Total_Omset']:>15,.0f}")

# Pola harian
print('\n' + '-'*70)
print('POLA HARIAN:')
print('-'*70)
dow_daily = df_harian.groupby('Hari').agg(
    Total_Omset=('Omset', 'sum'),
    Rata_rata=('Omset', 'mean')
).reindex(day_order)

for hari, row in dow_daily.iterrows():
    print(f"{hari:10s}: Rata-rata Rp {row['Rata_rata']:>12,.0f} | Total: Rp {row['Total_Omset']:>15,.0f}")

---
## 9. DAMPAK HARI LIBUR NASIONAL

In [ ]:
# Dampak hari libur
print('='*70)
print('DAMPAK HARI LIBUR NASIONAL Q1 2026')
print('='*70)

holidays = {
    '2026-01-01': 'Tahun Baru',
    '2026-01-29': "Isra Mi'raj",
    '2026-02-17': 'Tahun Baru Imlek',
    '2026-03-19': 'Nyepi'
}

holiday_dates = [pd.Timestamp(d) for d in holidays.keys()]
normal_days = df_harian[
    (~df_harian['Tanggal'].isin(holiday_dates)) & 
    (~df_harian['Hari'].isin(['Sabtu', 'Minggu']))
]
avg_normal = normal_days.groupby('Tanggal')['Omset'].sum().mean()

print(f'\nRata-rata omset hari kerja normal: Rp {avg_normal:,.0f}')
print('\n' + '-'*70)

holiday_data = []
for date_str, name in holidays.items():
    date = pd.Timestamp(date_str)
    if date in df_harian['Tanggal'].values:
        holiday_omset = df_harian[df_harian['Tanggal'] == date]['Omset'].sum()
        penurunan = ((avg_normal - holiday_omset) / avg_normal * 100)
        print(f"{date_str} ({name:20s}): Rp {holiday_omset:>15,.0f} | Penurunan: {penurunan:.1f}%")
        holiday_data.append({'Tanggal': date_str, 'Libur': name, 'Omset': holiday_omset, 'Penurunan': penurunan})

print('\nInsight: Hari libur menyebabkan penurunan omset 45-49%.')

---
## 10. MIX PRODUK PER CABANG

In [ ]:
# Mix produk per cabang
print('='*70)
print('MIX PRODUK PER CABANG (Persentase Omset)')
print('='*70)

mix_data = df_harian.groupby(['Nama_Cabang', 'Kategori'])['Omset'].sum().reset_index()
total_per_branch = mix_data.groupby('Nama_Cabang')['Omset'].sum().reset_index()
total_per_branch.columns = ['Nama_Cabang', 'Total']
mix_data = mix_data.merge(total_per_branch, on='Nama_Cabang')
mix_data['Persentase'] = (mix_data['Omset'] / mix_data['Total'] * 100).round(2)

# Pivot untuk tampilan
mix_pivot = mix_data.pivot(index='Nama_Cabang', columns='Kategori', values='Persentase')
mix_pivot = mix_pivot.fillna(0)

print('\n' + mix_pivot.round(1).to_string())

---
## 11. EFISIENSI HARGA

In [ ]:
# Efisiensi harga
print('='*70)
print('EFISIENSI HARGA JUAL vs HARGA STANDAR')
print('='*70)

price_analysis = df_harian.groupby(['Nama_Standar', 'Kode_Standar']).agg(
    Harga_Jual_Rata2=('Harga_Satuan', 'mean'),
    Harga_Jual_Min=('Harga_Satuan', 'min'),
    Harga_Jual_Max=('Harga_Satuan', 'max'),
    Std_Deviasi=('Harga_Satuan', 'std')
).reset_index()

std_prices = df_mapping[['Kode_Standar', 'Nama_Standar', 'Harga_Standar']].drop_duplicates()
price_analysis = price_analysis.merge(std_prices, on=['Kode_Standar', 'Nama_Standar'])
price_analysis['Deviasi_Persen'] = ((price_analysis['Harga_Jual_Rata2'] - price_analysis['Harga_Standar']) / price_analysis['Harga_Standar'] * 100).round(2)

print('\n' + '-'*80)
for _, row in price_analysis.iterrows():
    status = 'Sesuai' if abs(row['Deviasi_Persen']) < 1 else 'Perlu Review'
    print(f"{row['Nama_Standar']:20s} | Standar: Rp {row['Harga_Standar']:>10,.0f} | Jual: Rp {row['Harga_Jual_Rata2']:>10,.0f} | Deviasi: {row['Deviasi_Persen']:+.2f}% | {status}")

---
## 12. PROYEKSI 1 BULAN KE DEPAN

In [ ]:
# Proyeksi 1 bulan ke depan (April 2026) dengan garis regresi lurus
from datetime import datetime

# Agregasi bulanan
monthly_omset = df_harian.groupby('Bulan')['Omset'].sum()

# Data harian untuk regresi
daily_omset = df_harian.groupby('Tanggal')['Omset'].sum().reset_index()
daily_omset['day_num'] = (daily_omset['Tanggal'] - daily_omset['Tanggal'].min()).dt.days

# Linear regression
coeffs = np.polyfit(daily_omset['day_num'], daily_omset['Omset'], 1)
trend_fn = np.poly1d(coeffs)

# Proyeksi 30 hari ke depan
last_date = daily_omset['Tanggal'].max()
forecast_dates = pd.date_range(start=last_date + timedelta(days=1), periods=30)
forecast_day_nums = np.array([(d - daily_omset['Tanggal'].min()).days for d in forecast_dates])
forecast_values = trend_fn(forecast_day_nums)
apr_proj = forecast_values.sum()

# Garis regresi lurus: nilai harian x jumlah hari per bulan
month_start_days = {'Januari': 0, 'Februari': 31, 'Maret': 59, 'April': 90}
month_n_days = {'Januari': 31, 'Februari': 28, 'Maret': 31, 'April': 30}

months = ['Januari', 'Februari', 'Maret', 'April\n(Proyeksi)']
bar_heights = [monthly_omset['Januari'], monthly_omset['Februari'], monthly_omset['Maret'], apr_proj]
trend_straight = [trend_fn(month_start_days[m]) * month_n_days[m] for m in ['Januari', 'Februari', 'Maret', 'April']]

print('='*70)
print('PROYEKSI PENJUALAN 1 BULAN KE DEPAN (April 2026)')
print('='*70)
print(f'\nJanuari 2026  : Rp {monthly_omset["Januari"]:>15,.0f}')
print(f'Februari 2026 : Rp {monthly_omset["Februari"]:>15,.0f}')
print(f'Maret 2026    : Rp {monthly_omset["Maret"]:>15,.0f}')
print(f'April 2026    : Rp {apr_proj:>15,.0f} (proyeksi)')
print(f'\nTrend Slope   : +Rp {coeffs[0]:,.0f}/hari')

# Bar chart + garis regresi lurus
colors = ['#636EFA', '#636EFA', '#636EFA', '#00CC96']

fig_proj = go.Figure()
fig_proj.add_trace(go.Bar(
    x=months, y=bar_heights, name='Omset',
    marker_color=colors,
    text=[f'Rp {v:,.0f}' for v in bar_heights],
    textposition='outside'
))
fig_proj.add_trace(go.Scatter(
    x=months, y=trend_straight, name='Garis Regresi',
    mode='lines',
    line=dict(color='#EF553B', width=3)
))
fig_proj.update_layout(
    title='PROYEKSI 1 BULAN KE DEPAN (April 2026) + Garis Regresi',
    xaxis_title='Bulan', yaxis_title='Omset (Rp)',
    height=450, legend=dict(x=0, y=1),
    yaxis=dict(rangemode='tozero')
)
fig_proj.show()
fig_proj.write_image('charts/09_proyeksi_april.png', scale=2)

---
## 13. VISUALISASI (5 Chart Utama)

In [ ]:
# VIS 1: Grouped bar omset per bulan per cabang
fig1 = px.bar(
    df_bulanan,
    x='Nama_Cabang',
    y='Total_Omset',
    color='Bulan',
    barmode='group',
    title='1. OMSET PER CABANG PER BULAN',
    labels={'Total_Omset': 'Total Omset (Rp)', 'Nama_Cabang': 'Cabang'}
)
fig1.update_layout(height=500)
fig1.show()
fig1.write_image('charts/01_omset_cabang_bulanan.png', scale=2)

In [ ]:
# VIS 2: Donut chart distribusi omset per cabang
branch_total = df_harian.groupby('Nama_Cabang')['Omset'].sum().sort_values(ascending=False)

fig2 = px.pie(
    values=branch_total.values,
    names=branch_total.index,
    title='2. DISTRIBUSI OMSET PER CABANG (Q1 2026)',
    hole=0.4
)
fig2.update_traces(textposition='inside', textinfo='percent+label')
fig2.update_layout(height=450)
fig2.show()
fig2.write_image('charts/02_donut_cabang.png', scale=2)

In [ ]:
# VIS 3: Horizontal bar top 10 produk
fig5 = px.bar(
    product_ranking.head(10),
    x='Total_Omset',
    y='Nama_Standar',
    color='Kategori',
    orientation='h',
    title='3. TOP 10 PRODUK TERLARIS (by Omset)',
    labels={'Total_Omset': 'Total Omset (Rp)', 'Nama_Standar': 'Produk'},
    text_auto='.2s'
)
fig5.update_layout(height=450, yaxis={'categoryorder': 'total ascending'})
fig5.show()
fig5.write_image('charts/03_top10_produk.png', scale=2)

In [ ]:
# VIS 4: Horizontal bar omset per kategori
cat_bar = df_harian.groupby('Kategori')['Omset'].sum().sort_values(ascending=True).reset_index()
cat_bar.columns = ['Kategori', 'Total_Omset']

fig_cat = px.bar(
    cat_bar,
    x='Total_Omset',
    y='Kategori',
    orientation='h',
    title='4. TOTAL OMSET PER KATEGORI',
    labels={'Total_Omset': 'Total Omset (Rp)', 'Kategori': 'Kategori'},
    color='Kategori',
    text_auto='.2s'
)
fig_cat.update_layout(height=400, showlegend=False)
fig_cat.show()
fig_cat.write_image('charts/04_kategori_bar.png', scale=2)

In [ ]:
# VIS 5: Tren omset harian (line chart)
daily_trend = df_harian.groupby('Tanggal')['Omset'].sum().reset_index()

fig_line = px.line(
    daily_trend,
    x='Tanggal',
    y='Omset',
    title='5. TREN OMSET HARIAN (SELURUH CABANG)',
    labels={'Omset': 'Omset (Rp)', 'Tanggal': 'Tanggal'}
)
fig_line.update_layout(height=400)
fig_line.show()
fig_line.write_image('charts/05_tren_harian.png', scale=2)

In [ ]:
print('='*70)
print('ANALISIS BULANAN SELESAI!')
print('='*70)
print('File: Analisis_Bulanan.ipynb')
print('Data: Data_Mentah_Harian.xlsx')